In [0]:
# Databricks notebook source

# MAGIC %md
# MAGIC # 🛡️ 05 · Risk Consolidation — Trust & Safety
# MAGIC
# MAGIC ### Consolidación de señales de riesgo por Review
# MAGIC
# MAGIC **Objetivo:** consolidar evaluaciones individuales de reglas Trust & Safety en una única evaluación por Review y por `Evaluation Run`.
# MAGIC
# MAGIC | Elemento | Valor |
# MAGIC |---|---|
# MAGIC | **Entrada principal** | `workspace.yelp_silver.slv_review_evaluation` |
# MAGIC | **Features** | `workspace.yelp_silver.slv_review_features` |
# MAGIC | **Salida** | `workspace.yelp_silver.slv_review_risk` |
# MAGIC | **Grano** | `1 Review × 1 Evaluation Run` |
# MAGIC | **Método** | Reglas heurísticas |
# MAGIC | **Decisión automatizada** | No |
# MAGIC | **Revisión humana** | Sí, cuando corresponda |
# MAGIC
# MAGIC > ⚠️ **Interpretación importante**  
# MAGIC > `weighted_risk_score` es un **score heurístico de priorización**.  
# MAGIC > No representa una probabilidad estadística de fraude y no confirma que una Review sea fraudulenta.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 1. Preparación
# MAGIC
# MAGIC Carga de librerías utilizadas para agregación, trazabilidad y generación de identificadores de ejecución.

# COMMAND ----------

# ============================================================
# 1. IMPORTS
# ============================================================

from pyspark.sql import functions as F

import uuid

# COMMAND ----------

# MAGIC %md
# MAGIC ## 2. Configuración del pipeline
# MAGIC
# MAGIC En esta sección se centralizan:
# MAGIC
# MAGIC - catálogo y schemas;
# MAGIC - tablas de entrada y salida;
# MAGIC - identificadores del pipeline;
# MAGIC - versión de la lógica de riesgo;
# MAGIC - metadata de gobierno;
# MAGIC - thresholds heurísticos.
# MAGIC
# MAGIC > Los thresholds permanecen versionados en código. Unity Catalog describe el producto, pero no reemplaza la configuración de la lógica.

# COMMAND ----------

# ============================================================
# 2. CONFIGURACIÓN
# ============================================================

CATALOG = "workspace"

SILVER_SCHEMA = "yelp_silver"
GOV_SCHEMA = "yelp_gov"

SILVER = f"{CATALOG}.{SILVER_SCHEMA}"
GOV = f"{CATALOG}.{GOV_SCHEMA}"


EVALUATION_TABLE = (
    f"{SILVER}.slv_review_evaluation"
)

FEATURE_TABLE = (
    f"{SILVER}.slv_review_features"
)

RISK_TABLE = (
    f"{SILVER}.slv_review_risk"
)

RUN_TABLE = (
    f"{GOV}.evaluation_run"
)


PIPELINE_NAME = (
    "05_risk_consolidation"
)

RISK_MODEL_VERSION = "1.0.0"

# ------------------------------------------------------------
# UNITY CATALOG / DATA GOVERNANCE
#
# Los owners se expresan como roles funcionales/técnicos y no
# como personas, evitando acoplar el notebook a nombres propios.
# ------------------------------------------------------------

DATA_DOMAIN = "trust_safety"
DATA_PRODUCT = "review_risk"
DATA_LAYER = "silver"
DATA_CLASSIFICATION = "internal"

TECHNICAL_OWNER = "data_engineering"
BUSINESS_OWNER = "trust_safety"
DATA_STEWARD = "data_governance"

RISK_METHODOLOGY = "heuristic_rules"
DECISION_TYPE = "prioritization"
AUTOMATED_DECISION = "false"
HUMAN_REVIEW_REQUIRED = "true"



# ------------------------------------------------------------
# OPCIONAL
#
# None:
# utiliza automáticamente el último Evaluation Run SUCCESS.
#
# También puedes colocar manualmente:
#
# EVALUATION_RUN_ID_OVERRIDE = "xxxx-xxxx-xxxx"
# ------------------------------------------------------------

EVALUATION_RUN_ID_OVERRIDE = None


# ------------------------------------------------------------
# THRESHOLDS HEURÍSTICOS
#
# Estos valores NO representan probabilidades estadísticas.
# Son criterios iniciales de priorización.
# ------------------------------------------------------------

LOW_THRESHOLD = 0.15
MEDIUM_THRESHOLD = 0.35
HIGH_THRESHOLD = 0.60


# ------------------------------------------------------------
# Investigación
# ------------------------------------------------------------

MIN_SIGNALS_INVESTIGATION = 2

MIN_SCORE_INVESTIGATION = 0.35


# ------------------------------------------------------------
# Prioridad P1
# ------------------------------------------------------------

P1_MIN_SIGNALS = 3
P1_MIN_SCORE = 0.60


RISK_CONSOLIDATION_ID = str(
    uuid.uuid4()
)


print("=" * 72)
print("YELPCONNECT - RISK CONSOLIDATION")
print("=" * 72)

print(
    f"Risk Consolidation ID : "
    f"{RISK_CONSOLIDATION_ID}"
)

print(
    f"Risk Model Version    : "
    f"{RISK_MODEL_VERSION}"
)

print("=" * 72)

# COMMAND ----------

# MAGIC %md
# MAGIC ### Parámetros heurísticos
# MAGIC
# MAGIC | Parámetro | Propósito |
# MAGIC |---|---|
# MAGIC | `LOW_THRESHOLD` | Inicio de señal de riesgo baja |
# MAGIC | `MEDIUM_THRESHOLD` | Inicio de señal de riesgo media |
# MAGIC | `HIGH_THRESHOLD` | Inicio de señal de riesgo alta |
# MAGIC | `MIN_SIGNALS_INVESTIGATION` | Mínimo de señales para investigación |
# MAGIC | `MIN_SCORE_INVESTIGATION` | Score mínimo alternativo para investigación |
# MAGIC | `P1_MIN_SIGNALS` | Señales mínimas para prioridad P1 |
# MAGIC | `P1_MIN_SCORE` | Score mínimo alternativo para prioridad P1 |
# MAGIC
# MAGIC Estos valores son criterios operativos iniciales y **no probabilidades estadísticas**.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 3. Validación de dependencias
# MAGIC
# MAGIC Antes de procesar se verifica que existan todas las tablas requeridas.  
# MAGIC El notebook falla explícitamente si falta una dependencia.

# COMMAND ----------

# ============================================================
# 3. VALIDAR TABLAS REQUERIDAS
# ============================================================

required_tables = [

    EVALUATION_TABLE,
    FEATURE_TABLE,
    RUN_TABLE
]


for table_name in required_tables:

    if not spark.catalog.tableExists(
        table_name
    ):

        raise Exception(
            f"""
            ❌ Tabla requerida no encontrada:

            {table_name}
            """
        )


print(
    "✔ Tablas requeridas disponibles."
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 4. Selección del Evaluation Run
# MAGIC
# MAGIC La ejecución puede trabajar de dos formas:
# MAGIC
# MAGIC 1. usando `EVALUATION_RUN_ID_OVERRIDE`, si se proporciona;
# MAGIC 2. seleccionando automáticamente el último `Evaluation Run` con estado `SUCCESS`.
# MAGIC
# MAGIC Esto evita consolidar evaluaciones incompletas o fallidas.

# COMMAND ----------

# ============================================================
# 4. DETERMINAR EVALUATION RUN
# ============================================================

if EVALUATION_RUN_ID_OVERRIDE:

    EVALUATION_RUN_ID = (
        EVALUATION_RUN_ID_OVERRIDE
    )

    run_exists = (

        spark.table(
            RUN_TABLE
        )

        .filter(
            F.col(
                "evaluation_run_id"
            )
            ==
            EVALUATION_RUN_ID
        )

        .filter(
            F.col(
                "execution_status"
            )
            ==
            "SUCCESS"
        )

        .limit(1)

        .count()
    )


    if run_exists == 0:

        raise Exception(
            f"""
            ❌ Evaluation Run no encontrado
            o no tiene estado SUCCESS:

            {EVALUATION_RUN_ID}
            """
        )

else:

    latest_run = (

        spark.table(
            RUN_TABLE
        )

        .filter(
            F.col(
                "execution_status"
            )
            ==
            "SUCCESS"
        )

        .orderBy(
            F.desc(
                "started_at"
            )
        )

        .select(
            "evaluation_run_id"
        )

        .first()
    )


    if latest_run is None:

        raise Exception(
            """
            ❌ No existe un Evaluation Run
            con estado SUCCESS.
            """
        )


    EVALUATION_RUN_ID = (
        latest_run.evaluation_run_id
    )


print(
    f"✔ Evaluation Run seleccionado: "
    f"{EVALUATION_RUN_ID}"
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 5. Carga de evaluaciones
# MAGIC
# MAGIC Se filtran únicamente las evaluaciones pertenecientes al `Evaluation Run` seleccionado y se calculan métricas básicas de control.

# COMMAND ----------

# ============================================================
# 5. CARGAR EVALUACIONES DEL RUN
# ============================================================

evaluations = (

    spark.table(
        EVALUATION_TABLE
    )

    .filter(
        F.col(
            "evaluation_run_id"
        )
        ==
        EVALUATION_RUN_ID
    )
)


evaluation_count = (
    evaluations.count()
)


review_count = (

    evaluations

    .select(
        "review_id"
    )

    .distinct()

    .count()
)


expected_rule_count = (

    evaluations

    .select(
        "rule_version_id"
    )

    .distinct()

    .count()
)


print(
    f"Evaluaciones     : {evaluation_count:,}"
)

print(
    f"Reviews          : {review_count:,}"
)

print(
    f"Reglas del Run   : {expected_rule_count}"
)


if evaluation_count == 0:

    raise Exception(
        """
        ❌ El Evaluation Run seleccionado
        no contiene evaluaciones.
        """
    )

# COMMAND ----------

# MAGIC %md
# MAGIC ## 6. Validación del grano
# MAGIC
# MAGIC Antes de consolidar se comprueba que cada Review tenga exactamente una evaluación por `rule_version_id`.
# MAGIC
# MAGIC > ✅ **Regla esperada:** `1 Review × N reglas únicas`
# MAGIC >
# MAGIC > Si existen reglas duplicadas o faltantes, el proceso se detiene.

# COMMAND ----------

# ============================================================
# 6. VALIDAR REVIEW × REGLAS
# ============================================================

grain_validation = (

    evaluations

    .groupBy(
        "review_id"
    )

    .agg(

        F.count("*")
        .alias(
            "evaluation_count"
        ),

        F.countDistinct(
            "rule_version_id"
        )
        .alias(
            "distinct_rule_count"
        )
    )
)


invalid_reviews = (

    grain_validation

    .filter(

        (
            F.col(
                "evaluation_count"
            )
            !=
            expected_rule_count
        )

        |

        (
            F.col(
                "distinct_rule_count"
            )
            !=
            expected_rule_count
        )
    )

    .count()
)


if invalid_reviews > 0:

    raise Exception(
        f"""
        ❌ Se encontraron {invalid_reviews:,}
        Reviews con evaluaciones duplicadas
        o reglas faltantes.

        No se realizará la consolidación.
        """
    )


print(
    "✔ Grano Review × Rule validado."
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 7. Consolidación de señales
# MAGIC
# MAGIC Para cada Review se consolidan:
# MAGIC
# MAGIC - número total de reglas;
# MAGIC - `SIGNAL`;
# MAGIC - `NO_SIGNAL`;
# MAGIC - `NOT_EVALUABLE`;
# MAGIC - score ponderado;
# MAGIC - reglas que generaron señales;
# MAGIC - versiones de reglas activadas;
# MAGIC - reglas no evaluables.

# COMMAND ----------

# ============================================================
# 7. CONSOLIDAR POR REVIEW
# ============================================================

risk_base = (

    evaluations

    .groupBy(
        "review_id"
    )

    .agg(

        # ----------------------------------------------------
        # Evaluation Run ID (constant — already filtered)
        # ----------------------------------------------------

        F.first(
            "evaluation_run_id"
        )
        .alias(
            "evaluation_run_id"
        ),


        # ----------------------------------------------------
        # Cantidad total de reglas
        # ----------------------------------------------------

        F.count("*")
        .alias(
            "rule_count"
        ),


        # ----------------------------------------------------
        # SIGNAL
        # ----------------------------------------------------

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "SIGNAL",

                F.lit(1)
            )

            .otherwise(
                F.lit(0)
            )
        )
        .cast("int")
        .alias(
            "signal_count"
        ),


        # ----------------------------------------------------
        # NO SIGNAL
        # ----------------------------------------------------

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "NO_SIGNAL",

                F.lit(1)
            )

            .otherwise(
                F.lit(0)
            )
        )
        .cast("int")
        .alias(
            "no_signal_count"
        ),


        # ----------------------------------------------------
        # NOT EVALUABLE
        # ----------------------------------------------------

        F.sum(

            F.when(
                F.col(
                    "evaluation_result"
                )
                ==
                "NOT_EVALUABLE",

                F.lit(1)
            )

            .otherwise(
                F.lit(0)
            )
        )
        .cast("int")
        .alias(
            "not_evaluable_count"
        ),


        # ----------------------------------------------------
        # SCORE
        #
        # Se suma el peso que ya asignó cada regla.
        # ----------------------------------------------------

        F.round(

            F.sum(
                "risk_score"
            ),

            4
        )
        .alias(
            "weighted_risk_score"
        ),


        # ----------------------------------------------------
        # Reglas que generaron SIGNAL
        # ----------------------------------------------------

        F.sort_array(

            F.collect_set(

                F.when(

                    F.col(
                        "evaluation_result"
                    )
                    ==
                    "SIGNAL",

                    F.col(
                        "rule_id"
                    )
                )
            )
        )
        .alias(
            "signal_rules"
        ),


        # ----------------------------------------------------
        # Versiones activadas
        # ----------------------------------------------------

        F.sort_array(

            F.collect_set(

                F.when(

                    F.col(
                        "evaluation_result"
                    )
                    ==
                    "SIGNAL",

                    F.col(
                        "rule_version_id"
                    )
                )
            )
        )
        .alias(
            "signal_rule_versions"
        ),


        # ----------------------------------------------------
        # Reglas no evaluables
        # ----------------------------------------------------

        F.sort_array(

            F.collect_set(

                F.when(

                    F.col(
                        "evaluation_result"
                    )
                    ==
                    "NOT_EVALUABLE",

                    F.col(
                        "rule_id"
                    )
                )
            )
        )
        .alias(
            "not_evaluable_rules"
        )
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 8. Cobertura de evaluación
# MAGIC
# MAGIC Se calcula la proporción de reglas efectivamente evaluables y se conserva explícitamente la existencia de casos `NOT_EVALUABLE`.
# MAGIC
# MAGIC Esto evita interpretar una Review parcialmente evaluada como si tuviera cobertura completa.

# COMMAND ----------

# ============================================================
# 8. COBERTURA DE EVALUACIÓN
# ============================================================

risk_base = (

    risk_base

    .withColumn(
        "evaluable_rule_count",

        F.col(
            "rule_count"
        )

        -

        F.col(
            "not_evaluable_count"
        )
    )

    .withColumn(
        "evaluation_coverage_pct",

        F.round(

            (
                F.col(
                    "evaluable_rule_count"
                )

                /

                F.col(
                    "rule_count"
                )
            )

            *
            100,

            2
        )
    )

    .withColumn(
        "has_incomplete_evaluation",

        F.col(
            "not_evaluable_count"
        ) > 0
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC # 🎯 Clasificación heurística de riesgo
# MAGIC
# MAGIC Esta parte transforma las señales consolidadas en variables operativas para análisis e investigación.
# MAGIC
# MAGIC La clasificación es **heurística**, no predictiva.

# COMMAND ----------

# MAGIC %md
# MAGIC ## 9. Nivel de señal
# MAGIC
# MAGIC Clasifica el score consolidado en `NONE`, `LOW`, `MEDIUM` o `HIGH`.

# COMMAND ----------

# ============================================================
# 9. SIGNAL LEVEL
# ============================================================

risk_base = (

    risk_base

    .withColumn(
        "signal_level",

        F.when(
            F.col(
                "weighted_risk_score"
            )
            >=
            HIGH_THRESHOLD,

            F.lit(
                "HIGH"
            )
        )

        .when(
            F.col(
                "weighted_risk_score"
            )
            >=
            MEDIUM_THRESHOLD,

            F.lit(
                "MEDIUM"
            )
        )

        .when(
            F.col(
                "weighted_risk_score"
            )
            >=
            LOW_THRESHOLD,

            F.lit(
                "LOW"
            )
        )

        .otherwise(
            F.lit(
                "NONE"
            )
        )
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 10. Requiere investigación
# MAGIC
# MAGIC Marca Reviews que cumplen criterios de señales o score mínimo.

# COMMAND ----------

# ============================================================
# 10. REQUIRES INVESTIGATION
# ============================================================

risk_base = (

    risk_base

    .withColumn(
        "requires_investigation",

        (
            F.col(
                "signal_count"
            )
            >=
            MIN_SIGNALS_INVESTIGATION
        )

        |

        (
            F.col(
                "weighted_risk_score"
            )
            >=
            MIN_SCORE_INVESTIGATION
        )
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 11. Prioridad de investigación
# MAGIC
# MAGIC Asigna prioridad operativa `P1`, `P2`, `P3` o `NONE`.

# COMMAND ----------

# ============================================================
# 11. PRIORIDAD DE INVESTIGACIÓN
# ============================================================

risk_base = (

    risk_base

    .withColumn(
        "investigation_priority",

        # ----------------------------------------------------
        # P1
        # ----------------------------------------------------

        F.when(

            (
                F.col(
                    "signal_count"
                )
                >=
                P1_MIN_SIGNALS
            )

            |

            (
                F.col(
                    "weighted_risk_score"
                )
                >=
                P1_MIN_SCORE
            ),

            F.lit(
                "P1"
            )
        )


        # ----------------------------------------------------
        # P2
        # ----------------------------------------------------

        .when(

            (
                F.col(
                    "signal_count"
                )
                >=
                MIN_SIGNALS_INVESTIGATION
            )

            |

            (
                F.col(
                    "weighted_risk_score"
                )
                >=
                MIN_SCORE_INVESTIGATION
            ),

            F.lit(
                "P2"
            )
        )


        # ----------------------------------------------------
        # P3
        # ----------------------------------------------------

        .when(

            F.col(
                "signal_count"
            )
            >=
            1,

            F.lit(
                "P3"
            )
        )


        # ----------------------------------------------------
        # NONE
        # ----------------------------------------------------

        .otherwise(
            F.lit(
                "NONE"
            )
        )
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 12. Clasificación funcional
# MAGIC
# MAGIC Genera una etiqueta funcional sin afirmar fraude confirmado.

# COMMAND ----------

# ============================================================
# 12. CLASIFICACIÓN FUNCIONAL
# ============================================================

risk_base = (

    risk_base

    .withColumn(
        "risk_classification",

        F.when(

            F.col(
                "requires_investigation"
            )
            ==
            True,

            F.lit(
                "POTENTIALLY_SUSPICIOUS"
            )
        )

        .when(

            F.col(
                "signal_count"
            )
            >
            0,

            F.lit(
                "SIGNAL_DETECTED"
            )
        )

        .otherwise(
            F.lit(
                "NO_RELEVANT_SIGNAL"
            )
        )
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC # 🧩 Enriquecimiento con contexto
# MAGIC
# MAGIC Se incorpora el snapshot de features utilizado para evaluar la Review, preservando trazabilidad hacia `feature_execution_id`.

# COMMAND ----------

# ============================================================
# 13. FEATURE SNAPSHOT
# ============================================================

feature_context = (

    spark.table(
        FEATURE_TABLE
    )

    .select(

        "review_id",
        "user_id",
        "business_id",

        "stars",
        "review_ts",

        "rating_deviation",
        "account_age_days",

        "user_reviews_30m",
        "user_reviews_1h",
        "user_reviews_24h",

        "user_review_count",

        "user_business_review_count",
        "user_business_concentration",

        "text_length",
        "text_repeat_count",

        "business_reviews_day",
        "business_avg_daily_reviews",
        "business_spike_ratio",

        "feature_execution_id"
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 14. Unión Risk + Feature Context
# MAGIC
# MAGIC Se combinan las métricas consolidadas con las variables explicativas y se agregan identificadores de trazabilidad del proceso.

# COMMAND ----------

# ============================================================
# 14. RISK + CONTEXTO
# ============================================================

review_risk = (

    risk_base

    .join(
        feature_context,
        "review_id",
        "left"
    )

    .withColumn(
        "risk_consolidation_id",

        F.lit(
            RISK_CONSOLIDATION_ID
        )
    )

    .withColumn(
        "risk_model_version",

        F.lit(
            RISK_MODEL_VERSION
        )
    )

    .withColumn(
        "risk_processed_at",

        F.current_timestamp()
    )

    .withColumn(
        "risk_pipeline_name",

        F.lit(
            PIPELINE_NAME
        )
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 15. Contrato final de salida
# MAGIC
# MAGIC La selección final agrupa las columnas en cinco bloques conceptuales:
# MAGIC
# MAGIC 1. identificación;
# MAGIC 2. información de Review;
# MAGIC 3. consolidación de reglas;
# MAGIC 4. riesgo y cobertura;
# MAGIC 5. trazabilidad.

# COMMAND ----------

# ============================================================
# 15. SELECCIÓN FINAL
# ============================================================

review_risk = (

    review_risk

    .select(

        # ----------------------------------------------------
        # Identificación
        # ----------------------------------------------------

        "review_id",
        "user_id",
        "business_id",

        "evaluation_run_id",


        # ----------------------------------------------------
        # Review
        # ----------------------------------------------------

        "review_ts",
        "stars",


        # ----------------------------------------------------
        # Consolidación de reglas
        # ----------------------------------------------------

        "rule_count",
        "evaluable_rule_count",
        "signal_count",
        "no_signal_count",
        "not_evaluable_count",

        "signal_rules",
        "signal_rule_versions",
        "not_evaluable_rules",


        # ----------------------------------------------------
        # Riesgo
        # ----------------------------------------------------

        "weighted_risk_score",
        "signal_level",

        "requires_investigation",
        "investigation_priority",

        "risk_classification",


        # ----------------------------------------------------
        # Cobertura / explicabilidad
        # ----------------------------------------------------

        "evaluation_coverage_pct",
        "has_incomplete_evaluation",


        # ----------------------------------------------------
        # Features principales
        # ----------------------------------------------------

        "rating_deviation",
        "account_age_days",

        "user_reviews_30m",
        "user_reviews_1h",
        "user_reviews_24h",

        "user_review_count",

        "user_business_review_count",
        "user_business_concentration",

        "text_length",
        "text_repeat_count",

        "business_reviews_day",
        "business_avg_daily_reviews",
        "business_spike_ratio",


        # ----------------------------------------------------
        # Trazabilidad
        # ----------------------------------------------------

        "feature_execution_id",

        "risk_consolidation_id",
        "risk_model_version",
        "risk_processed_at",
        "risk_pipeline_name"
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 16. Validación del grano final
# MAGIC
# MAGIC La salida debe cumplir estrictamente:
# MAGIC
# MAGIC > **1 fila por Review para el Evaluation Run seleccionado**
# MAGIC
# MAGIC Cualquier desviación detiene el proceso antes de persistir.

# COMMAND ----------

# ============================================================
# 16. VALIDAR GRANO FINAL
# ============================================================

risk_row_count = (
    review_risk.count()
)


risk_review_count = (

    review_risk

    .select(
        "review_id"
    )

    .distinct()

    .count()
)


print(
    f"Reviews origen       : {review_count:,}"
)

print(
    f"Risk rows generadas  : {risk_row_count:,}"
)

print(
    f"Review ID únicos     : {risk_review_count:,}"
)


if (
    risk_row_count != review_count

    or

    risk_review_count != review_count
):

    raise Exception(
        """
        ❌ No se cumple el grano:

        1 Review × 1 Evaluation Run
        """
    )


print(
    "✔ Grano de Risk Consolidation validado."
)

# COMMAND ----------

# MAGIC %md
# MAGIC # 💾 Persistencia Delta
# MAGIC
# MAGIC La tabla de riesgo se persiste en Delta usando `MERGE`.
# MAGIC
# MAGIC ### Clave lógica de idempotencia
# MAGIC
# MAGIC `review_id + evaluation_run_id`
# MAGIC
# MAGIC Esto permite volver a ejecutar el mismo run sin duplicar Reviews.

# COMMAND ----------

# ============================================================
# 17. CREAR TABLA DESTINO
# ============================================================

spark.sql(
    f"""
    CREATE TABLE IF NOT EXISTS
    {RISK_TABLE}
    (
        review_id                     STRING,
        user_id                       STRING,
        business_id                   STRING,

        evaluation_run_id             STRING,

        review_ts                     TIMESTAMP,
        stars                         DOUBLE,

        rule_count                    BIGINT,
        evaluable_rule_count          BIGINT,
        signal_count                  INT,
        no_signal_count               INT,
        not_evaluable_count           INT,

        signal_rules                  ARRAY<STRING>,
        signal_rule_versions          ARRAY<STRING>,
        not_evaluable_rules           ARRAY<STRING>,

        weighted_risk_score           DOUBLE,
        signal_level                  STRING,

        requires_investigation        BOOLEAN,
        investigation_priority        STRING,

        risk_classification           STRING,

        evaluation_coverage_pct       DOUBLE,
        has_incomplete_evaluation     BOOLEAN,

        rating_deviation              DOUBLE,
        account_age_days              INT,

        user_reviews_30m              BIGINT,
        user_reviews_1h               BIGINT,
        user_reviews_24h              BIGINT,

        user_review_count             BIGINT,

        user_business_review_count    BIGINT,
        user_business_concentration   DOUBLE,

        text_length                   INT,
        text_repeat_count             BIGINT,

        business_reviews_day          BIGINT,
        business_avg_daily_reviews    DOUBLE,
        business_spike_ratio          DOUBLE,

        feature_execution_id          STRING,

        risk_consolidation_id         STRING,
        risk_model_version            STRING,
        risk_processed_at             TIMESTAMP,
        risk_pipeline_name            STRING
    )
    USING DELTA
    """
)

# COMMAND ----------

# ============================================================
# 18. TEMP VIEW
# ============================================================

review_risk.createOrReplaceTempView(
    "src_review_risk"
)

# COMMAND ----------

# ============================================================
# 19. MERGE IDEMPOTENTE
# ============================================================

spark.sql(
    f"""
    MERGE INTO
        {RISK_TABLE} AS target

    USING
        src_review_risk AS source

    ON
        target.review_id =
            source.review_id

        AND

        target.evaluation_run_id =
            source.evaluation_run_id

    WHEN MATCHED THEN

        UPDATE SET *

    WHEN NOT MATCHED THEN

        INSERT *
    """
)


print(
    f"✔ Tabla actualizada: "
    f"{RISK_TABLE}"
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## 20. Validación de persistencia
# MAGIC
# MAGIC Se compara el número de Reviews persistidas contra el número de Reviews del run de origen.
# MAGIC
# MAGIC La persistencia se considera válida únicamente si ambos valores coinciden.

# COMMAND ----------

# ============================================================
# 20. VALIDAR RUN PERSISTIDO
# ============================================================

persisted_count = (

    spark.table(
        RISK_TABLE
    )

    .filter(
        F.col(
            "evaluation_run_id"
        )
        ==
        EVALUATION_RUN_ID
    )

    .count()
)


print(
    f"Persistidas para Run : "
    f"{persisted_count:,}"
)


if persisted_count != review_count:

    raise Exception(
        """
        ❌ El número de Reviews persistidas
        no coincide con la ejecución origen.
        """
    )

# COMMAND ----------

# MAGIC %md
# MAGIC # 🏛️ 21. Unity Catalog — Gobierno y Metadata
# MAGIC
# MAGIC La tabla `slv_review_risk` se documenta como un **producto de datos derivado** para priorización Trust & Safety.
# MAGIC
# MAGIC ### Principios de gobierno
# MAGIC
# MAGIC | Aspecto | Definición |
# MAGIC |---|---|
# MAGIC | **Data Domain** | `trust_safety` |
# MAGIC | **Data Product** | `review_risk` |
# MAGIC | **Layer** | `silver` |
# MAGIC | **Classification** | `internal` |
# MAGIC | **Risk Methodology** | `heuristic_rules` |
# MAGIC | **Automated Decision** | `false` |
# MAGIC | **Human Review Required** | `true` |
# MAGIC | **Technical Owner** | `data_engineering` |
# MAGIC | **Business Owner** | `trust_safety` |
# MAGIC | **Data Steward** | `data_governance` |
# MAGIC
# MAGIC > 🧭 Unity Catalog almacena semántica, ownership y gobernanza.  
# MAGIC > Los thresholds de riesgo continúan versionados en código/configuración.

# COMMAND ----------

# ============================================================
# 21.1 APLICAR METADATA DE UNITY CATALOG
# ============================================================

# Descripción semántica de la tabla
table_comment = (
    "Producto de datos Trust & Safety que consolida señales "
    "heurísticas por Review y Evaluation Run. "
    "weighted_risk_score se utiliza para priorización y no "
    "representa una probabilidad estadística ni confirma fraude. "
    "La salida no constituye una decisión automatizada final."
)

escaped_table_comment = table_comment.replace("'", "''")

spark.sql(
    f"""
    COMMENT ON TABLE {RISK_TABLE}
    IS '{escaped_table_comment}'
    """
)


# ------------------------------------------------------------
# TAGS A NIVEL TABLA
# ------------------------------------------------------------
#
# No declaramos contains_pii=false porque user_id es un
# identificador y su clasificación formal depende de la política
# de privacidad definida por el proyecto.
# ------------------------------------------------------------

spark.sql(
    f"""
    ALTER TABLE {RISK_TABLE}
    SET TAGS (
        'data_layer' = '{DATA_LAYER}',
        'medallion_layer' = '{DATA_LAYER}',
        'data_domain' = '{DATA_DOMAIN}',
        'data_product' = '{DATA_PRODUCT}',
        'pipeline' = '{PIPELINE_NAME}',
        'classification' = '{DATA_CLASSIFICATION}',
        'contains_derived_risk' = 'true',
        'contains_identifier' = 'true',
        'risk_score_type' = 'heuristic',
        'risk_methodology' = '{RISK_METHODOLOGY}',
        'decision_type' = '{DECISION_TYPE}',
        'automated_decision' = '{AUTOMATED_DECISION}',
        'human_review_required' = '{HUMAN_REVIEW_REQUIRED}',
        'source_system' = 'yelp',
        'technical_owner' = '{TECHNICAL_OWNER}',
        'business_owner' = '{BUSINESS_OWNER}',
        'data_steward' = '{DATA_STEWARD}'
    )
    """
)


# ------------------------------------------------------------
# COMENTARIOS SEMÁNTICOS DE COLUMNAS
# ------------------------------------------------------------

column_comments = {
    "review_id":
        "Identificador de la Review evaluada.",

    "user_id":
        "Identificador del usuario asociado a la Review.",

    "business_id":
        "Identificador del negocio asociado a la Review.",

    "evaluation_run_id":
        "Identificador del Evaluation Run que originó la consolidación.",

    "rule_count":
        "Cantidad total de reglas consideradas para la Review.",

    "evaluable_rule_count":
        "Cantidad de reglas que pudieron ser evaluadas.",

    "signal_count":
        "Cantidad de reglas cuyo resultado fue SIGNAL.",

    "no_signal_count":
        "Cantidad de reglas cuyo resultado fue NO_SIGNAL.",

    "not_evaluable_count":
        "Cantidad de reglas cuyo resultado fue NOT_EVALUABLE.",

    "signal_rules":
        "Conjunto de rule_id que generaron SIGNAL.",

    "signal_rule_versions":
        "Versiones de reglas que generaron SIGNAL.",

    "not_evaluable_rules":
        "Reglas que no pudieron ser evaluadas.",

    "weighted_risk_score":
        "Score heurístico consolidado para priorización. "
        "No representa una probabilidad estadística de fraude.",

    "signal_level":
        "Nivel heurístico derivado del score: NONE, LOW, MEDIUM o HIGH.",

    "requires_investigation":
        "Indica si la Review cumple criterios heurísticos "
        "para investigación adicional.",

    "investigation_priority":
        "Prioridad operativa de investigación: P1, P2, P3 o NONE.",

    "risk_classification":
        "Clasificación funcional derivada de señales. "
        "No constituye confirmación de fraude.",

    "evaluation_coverage_pct":
        "Porcentaje de reglas que pudieron ser evaluadas.",

    "has_incomplete_evaluation":
        "Indica si una o más reglas resultaron NOT_EVALUABLE.",

    "feature_execution_id":
        "Identificador de la ejecución de Feature Engineering utilizada.",

    "risk_consolidation_id":
        "Identificador único de la ejecución de consolidación de riesgo.",

    "risk_model_version":
        "Versión de la lógica heurística de consolidación.",

    "risk_processed_at":
        "Timestamp de procesamiento de la consolidación.",

    "risk_pipeline_name":
        "Nombre del pipeline que produjo el registro."
}

for column_name, description in column_comments.items():

    escaped_description = description.replace("'", "''")

    spark.sql(
        f"""
        ALTER TABLE {RISK_TABLE}
        ALTER COLUMN {column_name}
        COMMENT '{escaped_description}'
        """
    )


# ------------------------------------------------------------
# TAGS A NIVEL COLUMNA
# ------------------------------------------------------------

column_tags = {
    "review_id": {
        "semantic_type": "identifier"
    },
    "user_id": {
        "semantic_type": "identifier"
    },
    "business_id": {
        "semantic_type": "identifier"
    },
    "evaluation_run_id": {
        "semantic_type": "execution_identifier"
    },
    "weighted_risk_score": {
        "semantic_type": "derived_risk_score",
        "risk_metric": "true"
    },
    "signal_level": {
        "semantic_type": "risk_level"
    },
    "requires_investigation": {
        "semantic_type": "investigation_flag"
    },
    "investigation_priority": {
        "semantic_type": "operational_priority"
    },
    "risk_classification": {
        "semantic_type": "risk_classification"
    },
    "evaluation_coverage_pct": {
        "semantic_type": "quality_metric"
    },
    "risk_model_version": {
        "semantic_type": "model_version"
    }
}

for column_name, tags in column_tags.items():

    tags_sql = ", ".join(
        [
            f"'{tag_name}' = '{tag_value}'"
            for tag_name, tag_value in tags.items()
        ]
    )

    spark.sql(
        f"""
        ALTER TABLE {RISK_TABLE}
        ALTER COLUMN {column_name}
        SET TAGS ({tags_sql})
        """
    )


print("✔ Metadata de Unity Catalog aplicada.")
print(f"  Tabla            : {RISK_TABLE}")
print(f"  Data Domain      : {DATA_DOMAIN}")
print(f"  Data Product     : {DATA_PRODUCT}")
print(f"  Technical Owner  : {TECHNICAL_OWNER}")
print(f"  Business Owner   : {BUSINESS_OWNER}")
print(f"  Data Steward     : {DATA_STEWARD}")


# COMMAND ----------

# MAGIC %md
# MAGIC ### 21.2 Validación de metadata
# MAGIC
# MAGIC La siguiente celda valida que los **tags de tabla** hayan quedado registrados
# MAGIC en `INFORMATION_SCHEMA`.
# MAGIC
# MAGIC Si devuelve filas, el tagging fue aplicado correctamente.

# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT
            tag_name,
            tag_value
        FROM {CATALOG}.information_schema.table_tags
        WHERE schema_name = '{SILVER_SCHEMA}'
          AND table_name = 'slv_review_risk'
        ORDER BY tag_name
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ### 21.3 Validación de tags de columnas
# MAGIC
# MAGIC Se valida la metadata semántica aplicada a las columnas críticas.

# COMMAND ----------

display(
    spark.sql(
        f"""
        SELECT
            column_name,
            tag_name,
            tag_value
        FROM {CATALOG}.information_schema.column_tags
        WHERE schema_name = '{SILVER_SCHEMA}'
          AND table_name = 'slv_review_risk'
        ORDER BY column_name, tag_name
        """
    )
)


# COMMAND ----------

# MAGIC %md
# MAGIC ### 21.4 Resultado esperado
# MAGIC
# MAGIC En **Catalog Explorer** deben quedar visibles:
# MAGIC
# MAGIC - descripción de la tabla;
# MAGIC - tags de dominio, producto, capa y gobierno;
# MAGIC - comentarios en columnas críticas;
# MAGIC - tags semánticos en identificadores y campos de riesgo.
# MAGIC
# MAGIC > **Importante:** el `Owner` nativo de Unity Catalog es independiente
# MAGIC > de los tags `technical_owner`, `business_owner` y `data_steward`.

# COMMAND ----------

# MAGIC %md
# MAGIC # 📊 Monitoreo del resultado
# MAGIC
# MAGIC Las siguientes vistas permiten inspeccionar rápidamente:
# MAGIC
# MAGIC - distribución de niveles de riesgo;
# MAGIC - distribución de prioridades;
# MAGIC - casos que requieren investigación;
# MAGIC - Reviews con señales múltiples;
# MAGIC - KPIs de ejecución.

# COMMAND ----------

# MAGIC %md
# MAGIC ## Distribución de niveles de riesgo

# COMMAND ----------

display(

    review_risk

    .groupBy(
        "signal_level"
    )

    .agg(

        F.count("*")
        .alias(
            "reviews"
        ),

        F.round(

            F.avg(
                "weighted_risk_score"
            ),

            4
        )
        .alias(
            "avg_score"
        )
    )

    .orderBy(

        F.when(
            F.col(
                "signal_level"
            )
            ==
            "HIGH",
            1
        )

        .when(
            F.col(
                "signal_level"
            )
            ==
            "MEDIUM",
            2
        )

        .when(
            F.col(
                "signal_level"
            )
            ==
            "LOW",
            3
        )

        .otherwise(
            4
        )
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Distribución de prioridades

# COMMAND ----------

display(

    review_risk

    .groupBy(
        "investigation_priority"
    )

    .count()

    .orderBy(

        F.when(
            F.col(
                "investigation_priority"
            )
            ==
            "P1",
            1
        )

        .when(
            F.col(
                "investigation_priority"
            )
            ==
            "P2",
            2
        )

        .when(
            F.col(
                "investigation_priority"
            )
            ==
            "P3",
            3
        )

        .otherwise(
            4
        )
    )
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Casos prioritarios para investigación

# COMMAND ----------

display(

    review_risk

    .filter(
        F.col(
            "requires_investigation"
        )
        ==
        True
    )

    .select(

        "review_id",
        "user_id",
        "business_id",

        "signal_count",
        "signal_rules",

        "weighted_risk_score",
        "signal_level",

        "investigation_priority",

        "evaluation_coverage_pct",

        "rating_deviation",
        "account_age_days",
        "user_reviews_30m",
        "user_business_concentration",
        "text_repeat_count",
        "business_spike_ratio"
    )

    .orderBy(

        F.when(
            F.col(
                "investigation_priority"
            )
            ==
            "P1",
            1
        )

        .otherwise(
            2
        ),

        F.desc(
            "weighted_risk_score"
        ),

        F.desc(
            "signal_count"
        )
    )

    .limit(100)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Reviews con múltiples señales

# COMMAND ----------

display(

    review_risk

    .filter(
        F.col(
            "signal_count"
        )
        >=
        2
    )

    .select(

        "review_id",
        "signal_count",
        "signal_rules",

        "weighted_risk_score",
        "signal_level",

        "investigation_priority"
    )

    .orderBy(

        F.desc(
            "signal_count"
        ),

        F.desc(
            "weighted_risk_score"
        )
    )

    .limit(100)
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## KPIs de ejecución

# COMMAND ----------

# ============================================================
# 26. KPIs DE EJECUCIÓN
# ============================================================

execution_kpis = (

    review_risk

    .agg(

        F.count("*")
        .alias(
            "reviews_evaluated"
        ),


        F.sum(

            F.when(
                F.col(
                    "signal_count"
                )
                >
                0,

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_with_signal"
        ),


        F.sum(

            F.when(
                F.col(
                    "requires_investigation"
                )
                ==
                True,

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_for_investigation"
        ),


        F.sum(

            F.when(
                F.col(
                    "investigation_priority"
                )
                ==
                "P1",

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "priority_p1"
        ),


        F.sum(

            F.when(
                F.col(
                    "has_incomplete_evaluation"
                )
                ==
                True,

                1
            )

            .otherwise(
                0
            )
        )
        .alias(
            "reviews_incomplete_evaluation"
        )
    )
)


display(
    execution_kpis
)

# COMMAND ----------

# MAGIC %md
# MAGIC ## Resumen final

# COMMAND ----------

# ============================================================
# 27. RESUMEN FINAL
# ============================================================

kpis = (
    execution_kpis.first()
)


print()
print("=" * 72)
print("RISK CONSOLIDATION FINALIZADO")
print("=" * 72)

print(
    f"Evaluation Run            : "
    f"{EVALUATION_RUN_ID}"
)

print(
    f"Reviews evaluadas         : "
    f"{kpis['reviews_evaluated']:,}"
)

print(
    f"Reviews con señal         : "
    f"{kpis['reviews_with_signal']:,}"
)

print(
    f"Requieren investigación   : "
    f"{kpis['reviews_for_investigation']:,}"
)

print(
    f"Prioridad P1              : "
    f"{kpis['priority_p1']:,}"
)

print(
    f"Evaluación incompleta     : "
    f"{kpis['reviews_incomplete_evaluation']:,}"
)

print(
    f"Output                    : "
    f"{RISK_TABLE}"
)

print("=" * 72)

# COMMAND ----------

# MAGIC %md
# MAGIC # ✅ Definition of Done
# MAGIC
# MAGIC El notebook se considera completo cuando:
# MAGIC
# MAGIC - [ ] se seleccionó un `Evaluation Run` exitoso;
# MAGIC - [ ] se validó el grano `Review × Rule`;
# MAGIC - [ ] se consolidaron todas las reglas por Review;
# MAGIC - [ ] se generaron `signal_count`, `signal_rules` y `weighted_risk_score`;
# MAGIC - [ ] se generaron `signal_level`, `requires_investigation` e `investigation_priority`;
# MAGIC - [ ] se conserva explícitamente `NOT_EVALUABLE`;
# MAGIC - [ ] se calculó cobertura de evaluación;
# MAGIC - [ ] la persistencia es idempotente mediante `MERGE`;
# MAGIC - [ ] no se etiqueta fraude confirmado;
# MAGIC - [ ] la tabla tiene descripción en Unity Catalog;
# MAGIC - [ ] se aplicaron tags de dominio, producto, ownership y stewardship;
# MAGIC - [ ] las columnas críticas tienen metadata semántica;
# MAGIC - [ ] el score está declarado como heurístico;
# MAGIC - [ ] no existe una decisión automatizada final;
# MAGIC - [ ] la revisión humana está documentada como control.
